# Notebook 06 — DuckDB Scenario Database

Builds `scenarios.duckdb` from the 10,000 generated log-signatures produced by notebook 04/05.

## What this notebook does
1. Loads `cvae_generated_logsigs.parquet` and `cvae_s2_params.json`
2. Validates column consistency between generated and real log-signatures
3. Creates a DuckDB database with four tables:
   - `scenarios` — 10,000 synthetic scenarios (log-signatures + regime labels)
   - `regime_weights` — generation fractions vs real-world fractions for reweighting
   - `model_config` — full CVAE architecture and scaler config from `cvae_s2_params.json`
   - `validation_summary` — pass/fail summary from notebook 05
4. Runs integrity checks and prints a summary

## Downstream consumers
- **Notebook 07a** (gas storage stochastic control): queries scenarios by regime, loads CVAE decoder
- **Notebook 07d** (deep hedging): uses log-signatures as the information set $\mathcal{I}_t$

## Important: regime distribution
The 10,000 scenarios use **asymmetric generation fractions** (calm 30%, volatile 30%, spike 25%, negative 15%).  
This is intentional over-sampling of tail regimes for risk applications.  
The `regime_weights` table stores importance weights to reweight back to real-world frequencies  
if an **unbiased** simulation is required.

## Cell 1 — Install DuckDB and mount Drive

In [84]:
%pip install duckdb --quiet

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/energy_synthetic_data/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR: /content/drive/MyDrive/energy_synthetic_data/data


## Cell 2 — Imports

In [85]:
import json
import numpy as np
import pandas as pd
import duckdb
from pathlib import Path

print("duckdb version:", duckdb.__version__)
print("pandas version: ", pd.__version__)

duckdb version: 1.3.2
pandas version:  2.2.2


## Cell 3 — Load generated log-signatures and model config

In [86]:
# ── Generated scenarios ───────────────────────────────────────────────────────
gen_path = DATA_DIR / "cvae_generated_logsigs_s2.parquet"
df_gen   = pd.read_parquet(gen_path)
print(f"Loaded generated parquet: {df_gen.shape}")
print("Columns:", df_gen.columns.tolist()[:8], "...")  # first few
print("Regimes:", df_gen["regime"].value_counts().to_dict())

# ── Real log-signatures (for column cross-check only) ─────────────────────────
real_path = DATA_DIR / "logsigs_s2.parquet"
df_real   = pd.read_parquet(real_path)
print(f"\nLoaded real parquet:      {df_real.shape}")

# ── Model config ──────────────────────────────────────────────────────────────
with open(DATA_DIR / "cvae_s2_params.json") as f:
    cvae_s2_params = json.load(f)
print(f"\ncvae_s2_params keys: {list(cvae_s2_params.keys())}")
print(f"logsig_dim (params): {cvae_s2_params['logsig_dim']}")
print(f"cond_dim   (params): {cvae_s2_params['cond_dim']}")
print(f"best_epoch:          {cvae_s2_params['best_epoch']}")
print(f"best_val_loss:       {cvae_s2_params['best_val_loss']}")

Loaded generated parquet: (10000, 98)
Columns: ['sample_id', 'regime', 'ls_1', 'ls_2', 'ls_3', 'ls_4', 'ls_5', 'ls_6'] ...
Regimes: {'calm': 3000, 'volatile': 3000, 'spike': 2500, 'negative': 1500}

Loaded real parquet:      (6530, 138)

cvae_s2_params keys: ['regime_map', 'c_scalar_dims', 'c_regime_dims', 'cond_scalar_cols', 't_gen', 'latent_dim', 'hidden_dim', 'n_layers', 'cond_dim', 'logsig_dim', 'best_epoch', 'best_val_loss', 'collapse_at_best', 'X_mean', 'X_scale', 'C_mean', 'C_scale']
logsig_dim (params): 89
cond_dim   (params): 11
best_epoch:          979
best_val_loss:       0.8024202585220337


## Cell 4 — Column consistency check

Generated and real parquets must share the same 31 active ls_ column names.
Always take the intersection — never assume they are identical.

In [87]:
ls_cols_real = [c for c in df_real.columns if c.startswith("ls_")]
ls_cols_gen  = [c for c in df_gen.columns  if c.startswith("ls_")]
ls_cols      = [c for c in ls_cols_real if c in ls_cols_gen]  # intersection, real order
logsig_dim   = len(ls_cols)

print(f"ls_ cols in real parquet:      {len(ls_cols_real)}")
print(f"ls_ cols in generated parquet: {len(ls_cols_gen)}")
print(f"Intersection (active cols):    {logsig_dim}")

# Cross-check against cvae_s2_params
expected_dim = cvae_s2_params["logsig_dim"]
assert logsig_dim == expected_dim, (
    f"logsig_dim mismatch: intersection={logsig_dim}, cvae_s2_params={expected_dim}"
)
print(f"\nlogsig_dim matches cvae_s2_params: {logsig_dim} ✓")

# Warn if any columns are not shared
only_real = set(ls_cols_real) - set(ls_cols_gen)
only_gen  = set(ls_cols_gen)  - set(ls_cols_real)
if only_real:
    print(f"WARNING: ls_ cols only in real: {sorted(only_real)}")
if only_gen:
    print(f"WARNING: ls_ cols only in gen:  {sorted(only_gen)}")

ls_ cols in real parquet:      90
ls_ cols in generated parquet: 89
Intersection (active cols):    89

logsig_dim matches cvae_s2_params: 89 ✓


## Cell 5 — Prepare the scenarios DataFrame

Adds `scenario_id` (int, 0-indexed), `regime_id` (int mapping from regime string),
and selects only the columns we want in the database.

In [88]:
# The `REGIME_MAP` is available in `cvae_s2_params`
REGIME_MAP = cvae_s2_params["regime_map"]

# Discover scalar conditioning columns dynamically from the generated parquet
# Everything that isn't sample_id, regime, or ls_* is a conditioning scalar
METADATA_COLS = {"sample_id", "regime", "regime_id"}
SCALAR_COND_NAMES = [c for c in df_gen.columns
                     if not c.startswith("ls_") and c not in METADATA_COLS]
print(f"Scalar conditioning columns (auto-discovered): {SCALAR_COND_NAMES}")


# If scalar conditioning is required, these columns would need to be sourced elsewhere or included in the generated parquet.

# Build scenarios table — ids, regime, log-sig dims
df_scenarios = df_gen[["sample_id"] + ["regime"] + ls_cols].copy()
df_scenarios = df_scenarios.rename(columns={"sample_id": "scenario_id"})
df_scenarios["regime_id"] = df_scenarios["regime"].map(REGIME_MAP).astype(int)

df_scenarios = df_gen[["sample_id", "regime"] + SCALAR_COND_NAMES + ls_cols].copy()
df_scenarios = df_scenarios.rename(columns={"sample_id": "scenario_id"})
df_scenarios["regime_id"] = df_scenarios["regime"].map(REGIME_MAP).astype(int)
df_scenarios = df_scenarios[
    ["scenario_id", "regime", "regime_id"] + SCALAR_COND_NAMES + ls_cols
]

print(f"scenarios shape: {df_scenarios.shape}")
print(f"Columns (first 8): {df_scenarios.columns.tolist()[:8]}")
print(f"Regime counts:\n{df_scenarios['regime'].value_counts()}")
assert df_scenarios["scenario_id"].nunique() == len(df_scenarios), "Duplicate scenario_ids"
assert df_scenarios["regime_id"].notna().all(), "NaN regime_ids — check REGIME_MAP"
print("\nscenarios table validated ✓")

Scalar conditioning columns (auto-discovered): ['wind_prev_mean', 'nuclear_prev_mean', 'RV_prev', 'wind_var_prev_mean', 'ccgt_fraction_prev_mean', 'ocgt_active_prev_mean', 'har_rv_prev_mean']
scenarios shape: (10000, 99)
Columns (first 8): ['scenario_id', 'regime', 'regime_id', 'wind_prev_mean', 'nuclear_prev_mean', 'RV_prev', 'wind_var_prev_mean', 'ccgt_fraction_prev_mean']
Regime counts:
regime
calm        3000
volatile    3000
spike       2500
negative    1500
Name: count, dtype: int64

scenarios table validated ✓


## Cell 6 — Prepare regime_weights table

Documents the generation fractions and real-world fractions.
Importance weights = `real_world_fraction / generation_fraction`.
Multiply scenario weights by `importance_weight` to simulate unbiased draws.

In [89]:
# ── Generation fractions (asymmetric over-sampling of tail regimes) ────────────
TARGET_FRACTIONS = {
    "calm":     0.30,
    "volatile": 0.30,
    "spike":    0.25,
    "negative": 0.15,
}

# ── Real-world frequencies from returns.parquet (4-state regime labels) ────────
# These were computed in notebook 03 from 157,757 half-hourly observations.
# calm≈30%, volatile≈31%, spike≈28%, negative≈10%  (from notebook 05 cue doc)
REAL_WORLD_FRACTIONS = {
    "calm":     0.30,
    "volatile": 0.31,
    "spike":    0.28,
    "negative": 0.10,
}

# Compute actual counts from generated data
actual_counts = df_scenarios["regime"].value_counts().to_dict()
total_scenarios = len(df_scenarios)

rows = []
for regime in ["calm", "volatile", "spike", "negative"]:
    gen_frac  = TARGET_FRACTIONS[regime]
    real_frac = REAL_WORLD_FRACTIONS[regime]
    count     = actual_counts.get(regime, 0)
    # importance weight for reweighting to real-world distribution
    iw = real_frac / gen_frac if gen_frac > 0 else np.nan
    rows.append({
        "regime":               regime,
        "regime_id":            REGIME_MAP[regime],
        "generated_count":      count,
        "generated_fraction":   round(count / total_scenarios, 4),
        "target_fraction":      gen_frac,
        "real_world_fraction":  real_frac,
        "importance_weight":    round(iw, 4),
    })

df_regime_weights = pd.DataFrame(rows)
print(df_regime_weights.to_string(index=False))
print("\nNote: multiply scenario by importance_weight to recover real-world distribution.")

  regime  regime_id  generated_count  generated_fraction  target_fraction  real_world_fraction  importance_weight
    calm          0             3000                0.30             0.30                 0.30             1.0000
volatile          1             3000                0.30             0.30                 0.31             1.0333
   spike          2             2500                0.25             0.25                 0.28             1.1200
negative          3             1500                0.15             0.15                 0.10             0.6667

Note: multiply scenario by importance_weight to recover real-world distribution.


## Cell 7 — Prepare model_config table

Stores all CVAE architecture and scaler parameters as key-value pairs.
Excludes large array fields (X_mean, X_scale, pca components) which remain in cvae_s2_params.json.

In [90]:
# Scalar config fields — exclude large arrays
ARRAY_FIELDS = {
    "X_mean", "X_scale", "C_mean", "C_scale",
    "pca_lagged_mean", "pca_lagged_components"
}

config_rows = []
for key, value in cvae_s2_params.items():
    if key in ARRAY_FIELDS:
        # Summarise arrays rather than storing them in the DB
        arr = np.array(value)
        config_rows.append({
            "key":         key,
            "value":       f"array shape={arr.shape}",
            "description": "stored in cvae_s2_params.json",
        })
    else:
        config_rows.append({
            "key":         key,
            "value":       str(value),
            "description": "",
        })

# Add generation metadata
config_rows += [
    {"key": "n_scenarios",        "value": str(total_scenarios), "description": "total generated"},
    {"key": "generation_step",    "value": "notebook_04",        "description": "which notebook generated scenarios"},
    {"key": "validation_step",    "value": "notebook_05",        "description": "which notebook validated"},
    {"key": "sig_mmd_p_value",    "value": "0.435",              "description": "from notebook 05 permutation test"},
    {"key": "ks_pass_rate",       "value": "0.581",              "description": "18/31 dims pass KS at p>0.05"},
    {"key": "logsig_active_cols", "value": json.dumps(ls_cols),  "description": "31 active ls_ column names"},
]

df_model_config = pd.DataFrame(config_rows)
print(df_model_config[["key", "value"]].to_string(index=False))

               key                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    value
        regime_map                                                                                                                                                                          

## Cell 8 — Prepare validation_summary table

In [91]:
val_path = DATA_DIR / "validation_report_s2.json"

if val_path.exists():
    with open(val_path) as f:
        val_report = json.load(f)
    print("Loaded validation_report.json")
    val_rows = [
        {"metric": k, "result": str(v)} for k, v in val_report.items()
    ]
else:
    # Fallback: hard-code the known notebook 05 results from the cue doc
    print("validation_report.json not found — using cue-doc values")
    val_rows = [
        {"metric": "sig_mmd_squared",   "result": "0.000018"},
        {"metric": "sig_mmd_p_value",   "result": "0.435"},
        {"metric": "sig_mmd_pass",      "result": "True"},
        {"metric": "ks_dims_pass",      "result": "18"},
        {"metric": "ks_dims_total",     "result": "31"},
        {"metric": "ks_pass_rate",      "result": "0.581"},
        {"metric": "std_net_move",      "result": "PASS (ratio 0.93)"},
        {"metric": "skew_net_move",     "result": "FAIL (ratio artifact, tiny abs diff)"},
        {"metric": "kurtosis",          "result": "FAIL (sign flip, known VAE Gaussian limitation)"},
        {"metric": "mean_levy_area",    "result": "PASS (ratio 1.10)"},
        {"metric": "std_levy_area",     "result": "PASS (ratio 1.01)"},
        {"metric": "large_move_rate",   "result": "PASS (ratio 1.06)"},
        {"metric": "stylized_facts_pass", "result": "4/5"},
        {"metric": "notes",             "result": "kurtosis sign flip is expected VAE posterior compression; does not block downstream use"},
    ]

df_validation = pd.DataFrame(val_rows)
print(df_validation.to_string(index=False))

validation_report.json not found — using cue-doc values
             metric                                                                                  result
    sig_mmd_squared                                                                                0.000018
    sig_mmd_p_value                                                                                   0.435
       sig_mmd_pass                                                                                    True
       ks_dims_pass                                                                                      18
      ks_dims_total                                                                                      31
       ks_pass_rate                                                                                   0.581
       std_net_move                                                                       PASS (ratio 0.93)
      skew_net_move                                                    FAIL (rat

## Cell 9 — Create DuckDB and write all tables

In [92]:
db_path = DATA_DIR / "scenarios_s2.duckdb"

# Remove existing DB so we start clean
if db_path.exists():
    db_path.unlink()
    print(f"Removed existing {db_path.name}")

con = duckdb.connect(str(db_path))
print(f"Created {db_path}")

# ── 1. scenarios ──────────────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS scenarios")
con.register("df_scenarios", df_scenarios)
con.execute("""
    CREATE TABLE scenarios AS
    SELECT * FROM df_scenarios
""")
# Primary key constraint (DuckDB doesn't enforce but documents intent)
n_scen = con.execute("SELECT COUNT(*) FROM scenarios").fetchone()[0]
print(f"  scenarios rows: {n_scen}")

# ── 2. regime_weights ─────────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS regime_weights")
con.register("df_regime_weights", df_regime_weights)
con.execute("""
    CREATE TABLE regime_weights AS
    SELECT * FROM df_regime_weights
""")
print(f"  regime_weights rows: {con.execute('SELECT COUNT(*) FROM regime_weights').fetchone()[0]}")

# ── 3. model_config ───────────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS model_config")
con.register("df_model_config", df_model_config)
con.execute("""
    CREATE TABLE model_config AS
    SELECT * FROM df_model_config
""")
print(f"  model_config rows: {con.execute('SELECT COUNT(*) FROM model_config').fetchone()[0]}")

# ── 4. validation_summary ─────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS validation_summary")
con.register("df_validation", df_validation)
con.execute("""
    CREATE TABLE validation_summary AS
    SELECT * FROM df_validation
""")
print(f"  validation_summary rows: {con.execute('SELECT COUNT(*) FROM validation_summary').fetchone()[0]}")

print("\nAll tables written.")

Removed existing scenarios_s2.duckdb
Created /content/drive/MyDrive/energy_synthetic_data/data/scenarios_s2.duckdb
  scenarios rows: 10000
  regime_weights rows: 4
  model_config rows: 23
  validation_summary rows: 14

All tables written.


## Cell 10 — Add indexes for downstream query patterns

Notebooks 07a and 07d will filter by regime and sample random batches.
DuckDB doesn't have traditional B-tree indexes but we can create a
regime-partitioned view and verify query performance.

In [93]:
# Convenience view: scenarios joined with importance weights
# Downstream notebooks can query this view and get reweighting info for free
con.execute("DROP VIEW IF EXISTS scenarios_weighted")
con.execute("""
    CREATE VIEW scenarios_weighted AS
    SELECT
        s.*,
        rw.real_world_fraction,
        rw.importance_weight
    FROM scenarios s
    LEFT JOIN regime_weights rw
        ON s.regime = rw.regime
""")
print("Created view: scenarios_weighted")

# Spot-check: count per regime in the view
regime_check = con.execute("""
    SELECT regime, COUNT(*) AS n, AVG(importance_weight) AS iw
    FROM scenarios_weighted
    GROUP BY regime
    ORDER BY regime
""").df()
print(regime_check.to_string(index=False))

Created view: scenarios_weighted
  regime    n     iw
    calm 3000 1.0000
negative 1500 0.6667
   spike 2500 1.1200
volatile 3000 1.0333


## Cell 11 — Integrity checks

In [94]:
print("=" * 55)
print("INTEGRITY CHECKS")
print("=" * 55)

# 1. Row count
n = con.execute("SELECT COUNT(*) FROM scenarios").fetchone()[0]
assert n == 10_000, f"Expected 10,000 scenarios, got {n}"
print(f"[PASS] scenarios count = {n}")

# 2. No NULL log-sig values
null_check = con.execute(f"""
    SELECT COUNT(*) FROM scenarios
    WHERE {' OR '.join([f"{c} IS NULL" for c in ls_cols])}
""").fetchone()[0]
assert null_check == 0, f"{null_check} rows have NULL log-sig values"
print(f"[PASS] No NULL log-sig values")

# 3. scenario_id is unique
n_unique = con.execute("SELECT COUNT(DISTINCT scenario_id) FROM scenarios").fetchone()[0]
assert n_unique == n, "Duplicate scenario_ids"
print(f"[PASS] scenario_id unique ({n_unique})")

# 4. regime_id consistent with regime string
inconsistent = con.execute("""
    SELECT COUNT(*) FROM scenarios s
    JOIN regime_weights rw ON s.regime = rw.regime
    WHERE s.regime_id != rw.regime_id
""").fetchone()[0]
assert inconsistent == 0, f"{inconsistent} rows have inconsistent regime_id"
print(f"[PASS] regime_id consistent with regime string")

# 5. All 4 regimes present
regimes_present = con.execute(
    "SELECT DISTINCT regime FROM scenarios ORDER BY regime"
).fetchall()
regimes_present = {r[0] for r in regimes_present}
assert regimes_present == {"calm", "volatile", "spike", "negative"}, \
    f"Missing regimes: {regimes_present}"
print(f"[PASS] All 4 regimes present: {sorted(regimes_present)}")

# 6. logsig_dim matches expected
n_ls_cols = len([c for c in con.execute("DESCRIBE scenarios").df()["column_name"]
                 if str(c).startswith("ls_")])
assert n_ls_cols == logsig_dim, f"Expected {logsig_dim} ls_ cols, got {n_ls_cols}"
print(f"[PASS] logsig_dim = {n_ls_cols} ls_ columns")

# 7. Table list
tables = con.execute("SHOW TABLES").fetchall()
table_names = {t[0] for t in tables}
expected_tables = {"scenarios", "regime_weights", "model_config", "validation_summary"}
assert expected_tables.issubset(table_names), f"Missing tables: {expected_tables - table_names}"
print(f"[PASS] All tables present: {sorted(table_names)}")

print("\nAll integrity checks passed ✓")

INTEGRITY CHECKS
[PASS] scenarios count = 10000
[PASS] No NULL log-sig values
[PASS] scenario_id unique (10000)
[PASS] regime_id consistent with regime string
[PASS] All 4 regimes present: ['calm', 'negative', 'spike', 'volatile']
[PASS] logsig_dim = 89 ls_ columns
[PASS] All tables present: ['df_model_config', 'df_regime_weights', 'df_scenarios', 'df_validation', 'model_config', 'regime_weights', 'scenarios', 'scenarios_weighted', 'validation_summary']

All integrity checks passed ✓


## Cell 12 — Diagnostic queries (for visual inspection)

In [95]:
print("── Regime distribution ──────────────────────────────────────")
print(con.execute("""
    SELECT
        regime,
        COUNT(*)                             AS n,
        ROUND(100.0 * COUNT(*) / 10000, 1)   AS pct,
        real_world_fraction * 100            AS real_world_pct,
        ROUND(importance_weight, 3)          AS iw
    FROM scenarios_weighted
    GROUP BY regime, real_world_fraction, importance_weight
    ORDER BY regime
""").df().to_string(index=False))

print("\n── Sample spike scenarios (first 3) ────────────────────────")
first_cols = ["scenario_id", "regime", "regime_id"] + ls_cols[:4]
sample_q = con.execute(f"""
    SELECT {', '.join(first_cols)}
    FROM scenarios
    WHERE regime = 'spike'
    LIMIT 3
""").df()
print(sample_q.to_string(index=False))

print("\n── Log-sig summary stats (spike regime, first ls_ col) ──────")
first_ls = ls_cols[0]
print(con.execute(f"""
    SELECT
        regime,
        ROUND(AVG({first_ls}), 4)    AS mean,
        ROUND(STDDEV({first_ls}), 4) AS std,
        ROUND(MIN({first_ls}), 4)    AS min,
        ROUND(MAX({first_ls}), 4)    AS max
    FROM scenarios
    GROUP BY regime
    ORDER BY regime
""").df().to_string(index=False))

print("\n── Model config (scalar fields) ─────────────────────────────")
print(con.execute("""
    SELECT key, value FROM model_config
    WHERE key NOT LIKE 'logsig_active_cols'
      AND value NOT LIKE 'array%'
    ORDER BY key
""").df().to_string(index=False))

── Regime distribution ──────────────────────────────────────
  regime    n  pct  real_world_pct    iw
    calm 3000 30.0            30.0 1.000
negative 1500 15.0            10.0 0.667
   spike 2500 25.0            28.0 1.120
volatile 3000 30.0            31.0 1.033

── Sample spike scenarios (first 3) ────────────────────────
 scenario_id regime  regime_id      ls_1      ls_2     ls_3      ls_4
           0  spike          2 -1.686885 -1.760458 1.096915 -0.093860
           1  spike          2  1.846061  1.377850 1.222794  0.087634
           2  spike          2 -1.320323 -1.431283 0.196788 -1.246427

── Log-sig summary stats (spike regime, first ls_ col) ──────
  regime    mean    std     min    max
    calm -0.0422 1.1168 -5.9325 4.2434
negative -0.0295 1.3870 -4.8067 5.0533
   spike -0.0622 1.2723 -5.4443 4.9731
volatile -0.0879 1.0986 -4.0815 3.8134

── Model config (scalar fields) ─────────────────────────────
             key                                                      

## Cell 13 — Helper query patterns for downstream notebooks

These are the query patterns that notebooks 07a and 07d will use.
Run this cell to confirm they work correctly.

In [96]:
# ── Pattern 1: load all scenarios as numpy array ───────────────────────────────
# Used by 07a/07d to get the full scenario matrix for batch decoding
X_all = con.execute(f"""
    SELECT {', '.join(ls_cols)} FROM scenarios ORDER BY scenario_id
""").fetchnumpy()
X_matrix = np.stack([X_all[c] for c in ls_cols], axis=1)
print(f"Pattern 1 — full log-sig matrix: {X_matrix.shape}")
assert X_matrix.shape == (10_000, logsig_dim)

# ── Pattern 2: regime-filtered sample ─────────────────────────────────────────
# Used by 07a for regime-conditional gas storage valuation
def sample_regime(regime_name, n=100, seed=42):
    """Return n randomly sampled scenarios for a given regime."""
    df = con.execute(f"""
        SELECT {', '.join(['scenario_id', 'regime'] + ls_cols)}
        FROM scenarios
        WHERE regime = '{regime_name}'
        USING SAMPLE {n} ROWS
    """).df()
    return df

for r in ["calm", "volatile", "spike", "negative"]:
    df_r = sample_regime(r, n=50)
    print(f"Pattern 2 — sample_regime('{r}', n=50): {df_r.shape}")

# ── Pattern 3: reweighted sample ──────────────────────────────────────────────
# Used when unbiased simulation is required (e.g. estimating expected P&L)
df_weighted = con.execute(f"""
    SELECT {', '.join(['scenario_id', 'regime', 'importance_weight'] + ls_cols)}
    FROM scenarios_weighted
    ORDER BY scenario_id
""").df()
print(f"\nPattern 3 — weighted DataFrame: {df_weighted.shape}")
print(f"  importance_weight range: [{df_weighted['importance_weight'].min():.3f}, {df_weighted['importance_weight'].max():.3f}]")

# ── Pattern 4: retrieve model config values for model loading ─────────────────
# Used by 07a/07d to reconstruct the CVAE decoder without hard-coding params
def get_config(key):
    row = con.execute(f"SELECT value FROM model_config WHERE key = '{key}'").fetchone()
    return row[0] if row else None

print(f"\nPattern 4 — get_config:")
for k in ["latent_dim", "hidden_dim", "n_layers", "cond_dim", "logsig_dim", "best_val_loss"]:
    print(f"  {k}: {get_config(k)}")

Pattern 1 — full log-sig matrix: (10000, 89)
Pattern 2 — sample_regime('calm', n=50): (23, 91)
Pattern 2 — sample_regime('volatile', n=50): (17, 91)
Pattern 2 — sample_regime('spike', n=50): (19, 91)
Pattern 2 — sample_regime('negative', n=50): (6, 91)

Pattern 3 — weighted DataFrame: (10000, 92)
  importance_weight range: [0.667, 1.120]

Pattern 4 — get_config:
  latent_dim: 16
  hidden_dim: 100
  n_layers: 1
  cond_dim: 11
  logsig_dim: 89
  best_val_loss: 0.8024202585220337


## Cell 14 — Close connection and print final summary

In [97]:
con.close()

db_size_mb = db_path.stat().st_size / 1024 / 1024

print("=" * 55)
print("NOTEBOOK 06 COMPLETE")
print("=" * 55)
print(f"Database:    {db_path}")
print(f"Size:        {db_size_mb:.1f} MB")
print("Tables:")
print(f"  scenarios           — 10,000 rows × {3 + logsig_dim} cols")
print( "  regime_weights      — 4 rows (generation + real-world fractions)")
print(f"  model_config        — {len(df_model_config)} key-value pairs")
print(f"  validation_summary  — {len(df_validation)} metrics")
print("Views:")
print( "  scenarios_weighted  — scenarios joined with importance weights")
print("\nRegime distribution (generated):")
for r in ["calm", "volatile", "spike", "negative"]:
    gen_frac  = TARGET_FRACTIONS[r]
    real_frac = REAL_WORLD_FRACTIONS[r]
    print(f"  {r:<10} gen={gen_frac*100:.0f}%  real={real_frac*100:.0f}%  iw={real_frac/gen_frac:.2f}")
print("\nNext step: notebook 07a (gas storage stochastic control)")
print("           notebook 07d (deep hedging)")

NOTEBOOK 06 COMPLETE
Database:    /content/drive/MyDrive/energy_synthetic_data/data/scenarios_s2.duckdb
Size:        9.3 MB
Tables:
  scenarios           — 10,000 rows × 92 cols
  regime_weights      — 4 rows (generation + real-world fractions)
  model_config        — 23 key-value pairs
  validation_summary  — 14 metrics
Views:
  scenarios_weighted  — scenarios joined with importance weights

Regime distribution (generated):
  calm       gen=30%  real=30%  iw=1.00
  volatile   gen=30%  real=31%  iw=1.03
  spike      gen=25%  real=28%  iw=1.12
  negative   gen=15%  real=10%  iw=0.67

Next step: notebook 07a (gas storage stochastic control)
           notebook 07d (deep hedging)
